# Ultimate RVC for Kaggle (Personal Fork)

GitHub Repository: [seruvseruv-boop/ultimate-rvc](https://github.com/seruvseruv-boop/ultimate-rvc)

### Features:
- **RMVPE+ Algorithm** (Pitch-dependent RMVPE: anti-crackling & crystal-clear singing voice)
- **Clean Instrumental & Backing Vocal Isolation** (Minusda qo'shiqchining ovozi qolib ketmaydi)
- **WAV Audio Output** (Lossless studio quality)
- **Kaggle GPU Support** (NVIDIA Tesla T4 x2 or P100 with 30 hours/week free quota)

--- 
> **Eslatma:** Kaggle o'ng panelida (Settings) **ACCELERATOR** qismida **GPU T4 x 2** yoki **GPU P100** tanlanganligiga hamda **INTERNET** yoqilganligiga (Internet on) ishonch hosil qiling!

In [ ]:
# 1. Initialize environment & dependencies
%pip install ipython-autotime pyngrok
%load_ext autotime

import codecs
import os
import time
from pathlib import Path
from urllib import request

from IPython.display import clear_output, display as i_display
from pyngrok import ngrok

print("Environment initialized successfully!")

In [ ]:
# 2. Clone personal repository
%cd /kaggle/working
!rm -rf HRVC
!git clone https://github.com/seruvseruv-boop/ultimate-rvc.git HRVC
%cd /kaggle/working/HRVC
print("Repository cloned successfully from seruvseruv-boop/ultimate-rvc!")

In [ ]:
# 3. Install packages via uv package manager
!apt-get update -qq && apt-get install -y -qq python3-dev unzip sox libsox-fmt-all
!curl -LsSf https://astral.sh/uv/0.9.11/install.sh | sh

# Add uv to PATH
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
os.environ["URVC_CONSOLE_LOG_LEVEL"] = "WARNING"
os.environ["MPLBACKEND"] = "Agg"

# Run prerequisite models download
!uv run --prerelease if-necessary-or-explicit --extra cuda ./src/ultimate_rvc/core/main.py
print("Dependencies and prerequisite models installed!")

In [ ]:
# 4. Launch Ultimate RVC Web UI with public Gradio link
# Choose sharing method: "gradio", "ngrok", or "cloudflared"
method = "gradio"
ngrok_token = ""  # Optional: fill your token if using ngrok

prerelease = "--prerelease if-necessary-or-explicit"
runpice = "./src/ultimate_rvc/web/main.py"

if method == "gradio":
    !uv run $prerelease --extra cuda $runpice --share
elif method == "ngrok":
    try:
        ngrok.set_auth_token(ngrok_token)
        ngrok.kill()
        tunnel = ngrok.connect(6969)
        print(f"Ngrok URL: {tunnel.public_url}")
        !uv run $prerelease --extra cuda $runpice --listen-port 6969
    except Exception as e:
        print(f"Error starting ngrok: {e}")
elif method == "cloudflared":
    !curl -LO https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    !rm -rf nohup.out
    !nohup cloudflared tunnel --url localhost:6969 &
    time.sleep(10)
    !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" nohup.out
    !uv run $prerelease --extra cuda $runpice --listen-port 6969
